# Day 5: Robustness checks

Would the results survive different analysis choices? This notebook tests the main ones. Nothing here changes the main analysis; each check asks "does the conclusion still hold if we do it another way?"

| Check | Question |
|---|---|
| R1 | Do the results depend on the one random train/test split? |
| R2 | Do they change if the survey weights are used? (This one matters: read it carefully.) |
| R3 | Can the model predict firms in a country it has never seen? |
| R4 | Does the purchasing/on-site contrast hold using only characteristics that cannot be a result of the energy decision? |
| R5 | Does the way SHAP values are added up into themes matter? |
| R6 | Do the class weights matter? |
| R7 | Does it matter that firm size and firm age are each entered twice? |

**How to run it:** *Runtime > Run all*, then upload `sme_energy_clean.csv`. It takes roughly 15 to 30 minutes on Colab because R1, R2 and R3 train several hundred models. Leave the tab open.

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score, confusion_matrix, precision_recall_fscore_support

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 100)
SEED = 42

NEEDED = ["sme_energy_clean.csv"]
if not all(os.path.exists(f) for f in NEEDED):
    try:
        from google.colab import files
        print("Choose", NEEDED[0]); files.upload()
    except ImportError:
        raise FileNotFoundError(f"Put {NEEDED} in the same folder as this notebook.")

df = pd.read_csv("sme_energy_clean.csv")
CLASS_NAMES = ["No renewable", "Purchased", "On-site"]
y = df["pathway_code"]
df["log_employees"] = np.log1p(df["employees"])
df["log_firm_age"] = np.log1p(df["firm_age_years"].clip(lower=0))

FIRM = ["nace_section", "size", "log_employees", "log_firm_age", "age_band", "turnover_band",
        "turnover_trend", "employment_trend", "hiring_difficulty", "sells"]
BEHAVIOUR = ([c for c in df.columns if c.startswith(("act_", "sup_", "ext_", "bar_", "help_"))]
             + ["n_other_actions", "n_barriers", "asked_followups", "investment", "climate_strategy",
                "green_products", "green_jobs", "recycled_materials", "lifespan_policy"])
SETS = {"A1 Firm": FIRM, "A2 Firm + country": FIRM + ["country"], "B All": FIRM + ["country"] + BEHAVIOUR}
NUM_WITH_GAPS = ["log_employees", "log_firm_age"]

def make_pre(columns):
    cat = [c for c in columns if not pd.api.types.is_numeric_dtype(df[c])]
    gaps = [c for c in columns if c in NUM_WITH_GAPS]
    num = [c for c in columns if pd.api.types.is_numeric_dtype(df[c]) and c not in NUM_WITH_GAPS]
    steps = [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
             ("gaps", Pipeline([("fill", SimpleImputer(strategy="median", add_indicator=True)), ("scale", StandardScaler())]), gaps)]
    if num: steps.append(("num", StandardScaler(), num))
    return ColumnTransformer(steps)

# The settings selected by cross-validation on Day 2. They are held fixed in every check below.
# (See the "Cross-validation" sheet of Day2_results.xlsx.)
LOGIT_C = {"A1 Firm": 1.0, "A2 Firm + country": 0.1, "B All": 0.1}
RF_SET = {"A1 Firm": dict(min_samples_leaf=10, max_features="sqrt"), "A2 Firm + country": dict(min_samples_leaf=10, max_features="sqrt"),
          "B All": dict(min_samples_leaf=3, max_features=0.3)}
XGB_SET = {"A1 Firm": dict(max_depth=3, learning_rate=0.03, n_estimators=200, colsample_bytree=0.9, min_child_weight=5, subsample=0.8),
           "A2 Firm + country": dict(max_depth=3, learning_rate=0.03, n_estimators=400, colsample_bytree=0.9, min_child_weight=5, subsample=0.8),
           "B All": dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.6, min_child_weight=1, subsample=0.8)}
def new_logit(set_name, class_weight="balanced"):
    return LogisticRegression(max_iter=3000, C=LOGIT_C[set_name], class_weight=class_weight)
def new_rf(set_name):
    return RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED, **RF_SET[set_name])
def new_xgb(set_name="B All"):
    return xgb.XGBClassifier(objective="multi:softprob", tree_method="hist", n_jobs=2, random_state=SEED, verbosity=0, **XGB_SET[set_name])
def auc(y_true, proba, w=None):
    return roc_auc_score(y_true, proba, multi_class="ovr", sample_weight=w)

def theme_of(p):
    if p == "country": return "Country"
    if p in ("size", "log_employees"): return "Firm size"
    if p in ("turnover_band", "turnover_trend", "employment_trend"): return "Turnover and growth"
    if p in ("nace_section", "sells"): return "Sector and offer"
    if p in ("log_firm_age", "age_band"): return "Firm age"
    if p == "hiring_difficulty": return "Hiring difficulty"
    if p.startswith("act_") or p in ("n_other_actions", "asked_followups"): return "Other green actions"
    if p == "investment": return "Investment level"
    if p.startswith(("sup_", "ext_")): return "Support used"
    if p.startswith("bar_") or p == "n_barriers": return "Barriers met"
    if p.startswith("help_"): return "Support wanted"
    if p == "climate_strategy": return "Climate strategy"
    return "Green products, jobs, materials"

def shap_by_predictor(model, pre, X_enc, predictors):
    # Exact TreeSHAP on the model's raw class scores, summed back from one-hot columns to predictors.
    contrib = model.get_booster().predict(xgb.DMatrix(X_enc), pred_contribs=True)[:, :, :-1]
    out = np.zeros((X_enc.shape[0], 3, len(predictors)))
    for j, name in enumerate(pre.get_feature_names_out()):
        out[:, :, predictors.index(owner(name, predictors))] += contrib[:, :, j]
    return out
def theme_importance(shap, predictors, w=None):
    themes = sorted(set(theme_of(p) for p in predictors))
    st = np.zeros((shap.shape[0], 3, len(themes)))
    for j, p in enumerate(predictors):
        st[:, :, themes.index(theme_of(p))] += shap[:, :, j]
    return pd.DataFrame(np.average(np.abs(st), axis=0, weights=w).T, index=themes, columns=CLASS_NAMES)

def owner(name, predictors):
    # Which original predictor does an encoded column belong to?
    cat = [c for c in predictors if not pd.api.types.is_numeric_dtype(df[c])]
    block, rest = name.split("__", 1)
    return max((c for c in cat if rest.startswith(c + "_")), key=len) if block == "cat" else rest.replace("missingindicator_", "")
def linear_by_predictor(logit, pre, X_enc, centre, predictors):
    # The same quantity for a logistic regression: coefficient x (value - average value), on the model's raw class scores.
    lin = (X_enc - centre)[:, None, :] * logit.coef_[None, :, :]
    out = np.zeros((X_enc.shape[0], 3, len(predictors)))
    for j, name in enumerate(pre.get_feature_names_out()):
        out[:, :, predictors.index(owner(name, predictors))] += lin[:, :, j]
    return out
def balanced_weights(y_part, w_part):
    # Each firm's weight, rescaled so the three pathways carry equal total weight. With w = 1 this is the usual "balanced" class weight.
    w_part = np.asarray(w_part, dtype=float); y_part = np.asarray(y_part)
    class_total = pd.Series(w_part).groupby(y_part).sum()
    sw = w_part * w_part.sum() / (3 * class_total.loc[y_part].values)
    return sw / sw.mean()
WEIGHTINGS = {"No survey weights (main)": None, "Within-country weights": "weight_country", "Population weights": "weight_eu27"}
def survey_w(index, scheme):
    return np.ones(len(index)) if WEIGHTINGS[scheme] is None else df.loc[index, WEIGHTINGS[scheme]].values

def run_split(tr, te, set_name, scheme):
    # Fit logistic regression and XGBoost on one split under one weighting scheme; return scores and theme importances.
    cols = SETS[set_name]; pre = make_pre(cols); Xtr = pre.fit_transform(df.loc[tr, cols]); Xte = pre.transform(df.loc[te, cols])
    wtr, wte = survey_w(tr, scheme), survey_w(te, scheme)
    sw = balanced_weights(y.loc[tr], wtr)
    lg = LogisticRegression(max_iter=3000, C=LOGIT_C[set_name]).fit(Xtr, y.loc[tr], sample_weight=sw)
    xg = new_xgb(set_name).fit(Xtr, y.loc[tr], sample_weight=sw)
    out = {"proba": {"Logistic regression": lg.predict_proba(Xte), "XGBoost": xg.predict_proba(Xte)}}
    out["themes"] = {"XGBoost": theme_importance(shap_by_predictor(xg, pre, Xte, cols), cols, w=wte),
                     "Logistic regression": theme_importance(linear_by_predictor(lg, pre, Xte, np.average(Xtr, axis=0, weights=wtr), cols), cols, w=wte)}
    out["fitted"] = (pre, Xtr, Xte, lg, xg)
    return out

train_idx, test_idx = train_test_split(df.index, test_size=0.20, stratify=y, random_state=SEED)     # the Day 2 split
N_SPLITS = 20
SPLITS = [train_test_split(df.index, test_size=0.20, stratify=y, random_state=1000 + s) for s in range(N_SPLITS)]
print("Firms:", len(df))

## R1. Does the result depend on the one random split?

The main analysis used one 80/20 split. Here the split is redrawn 20 times with different random seeds. For each split the models are refitted and scored. If the conclusions are stable, the scores should vary little and the difference between XGBoost and logistic regression should stay small.

For the full predictor set we also recompute the theme importances on every split, to see whether the contrast between purchasing and on-site generation is stable.

In [ ]:
t0 = time.time()
rows, runs = [], {}          # runs[(weighting, split)] keeps the full-predictor results for R2
MAIN = "No survey weights (main)"
for i, (tr, te) in enumerate(SPLITS):
    for set_name in SETS:
        res = run_split(tr, te, set_name, MAIN)
        r = {"split": i, "set": set_name, **{m: auc(y.loc[te], p) for m, p in res["proba"].items()}}
        if set_name == "B All":
            pre, Xtr, Xte, _, _ = res["fitted"]
            r["Random forest"] = auc(y.loc[te], new_rf(set_name).fit(Xtr, y.loc[tr]).predict_proba(Xte))
            del res["fitted"]; runs[(MAIN, i)] = res
        rows.append(r)
splits = pd.DataFrame(rows)
splits["XGBoost minus logistic"] = splits["XGBoost"] - splits["Logistic regression"]
r1 = splits.groupby("set", sort=False).agg(
    logit_mean=("Logistic regression", "mean"), logit_sd=("Logistic regression", "std"),
    xgb_mean=("XGBoost", "mean"), xgb_sd=("XGBoost", "std"),
    rf_mean=("Random forest", "mean"), rf_sd=("Random forest", "std"),
    diff_mean=("XGBoost minus logistic", "mean"), diff_min=("XGBoost minus logistic", "min"),
    diff_max=("XGBoost minus logistic", "max"),
    xgb_higher_in=("XGBoost minus logistic", lambda d: int((d > 0).sum()))).round(4)
print(f"R1 took {(time.time() - t0) / 60:.1f} minutes"); r1

In [ ]:
THEMES = sorted(set(theme_of(p) for p in SETS["B All"]))
def stability(scheme, model):
    # Across the 20 splits: average importance for each pathway, and the on-site / purchased ratio.
    P = pd.DataFrame({i: runs[(scheme, i)]["themes"][model]["Purchased"] for i in range(N_SPLITS)})
    O = pd.DataFrame({i: runs[(scheme, i)]["themes"][model]["On-site"] for i in range(N_SPLITS)})
    R = O / P
    return pd.DataFrame({"Purchased": P.mean(axis=1), "On-site": O.mean(axis=1), "Ratio median": R.median(axis=1),
                         "Ratio min": R.min(axis=1), "Ratio max": R.max(axis=1), "Splits with ratio > 1": (R > 1).sum(axis=1)})
r1_shap = stability(MAIN, "XGBoost").sort_values("On-site", ascending=False).round(3)
r1_shap_logit = stability(MAIN, "Logistic regression").loc[r1_shap.index].round(3)
print("XGBoost"); display(r1_shap); print("Logistic regression"); r1_shap_logit

## R2. Do survey weights change the results?

This is the check that matters most, so it is done in three steps.

**Background.** The survey interviewed a similar number of firms in every country, whatever the country's size, and it interviewed far more small and medium-sized firms than their share of the population. Two sets of weights correct for this:

* **Within-country weights** correct the mix of firm sizes and sectors inside each country. Each country still counts about equally.
* **Population weights** (the column `weight_eu27`) do the same and also make each country count in proportion to its number of SMEs. A few large countries then dominate.

The main models used no weights, so they describe the sample as drawn: 20 countries counted about equally.

### Step 1. How uneven are the weights?

In [ ]:
def kish(w): return w.sum() ** 2 / (w ** 2).sum()
r2_weights = pd.DataFrame([{"Weight": name, "Firms": len(df), "Effective sample size": round(kish(df[col])),
                            "Share of total weight held by the top 10% of firms (%)": round(100 * df[col].nlargest(len(df) // 10).sum() / df[col].sum(), 1),
                            "Share held by micro firms (%)": round(100 * df.loc[df["size"].str.startswith("Micro"), col].sum() / df[col].sum(), 1)}
                           for name, col in [("Within-country", "weight_country"), ("Population", "weight_eu27")]])
country_weight = (100 * df.groupby("country")["weight_eu27"].sum() / df["weight_eu27"].sum()).sort_values(ascending=False)
print("Firms interviewed per country: %d to %d" % (df["country"].value_counts().min(), df["country"].value_counts().max()))
print("Share of the population weight held by the five largest countries: %.1f%% (%s)" % (country_weight.head(5).sum(), ", ".join(country_weight.head(5).index)))
r2_weights

The **effective sample size** says how many equally weighted firms would carry the same information. If it is far below the number of firms, a weighted model is being driven by a small number of heavily weighted firms and will be noisier.

### Step 2. A check that uses no model at all

How much does each pathway vary between countries? For each country we take the odds of being on the pathway rather than using no renewable energy, on a log scale (the same scale the models work on), and measure the spread across the 27 countries. The spread is computed twice: counting every country equally, and counting each country by its number of SMEs.

In [ ]:
pop_share = df.groupby("country")["weight_eu27"].sum(); pop_share = pop_share / pop_share.sum()
def country_spread(frame):
    w = frame["weight_country"]
    share = w.groupby([frame["country"], frame["pathway"]]).sum().unstack().pipe(lambda t: t.div(t.sum(axis=1), axis=0))
    lo = pd.DataFrame({"Purchased": np.log(share["Purchased"] / share["No renewable"]), "On-site": np.log(share["On-site"] / share["No renewable"])})
    equal = lo.std(ddof=0)
    centred = lo - (lo.mul(pop_share, axis=0)).sum()
    weighted = np.sqrt((centred ** 2).mul(pop_share, axis=0).sum())
    return equal, weighted
equal, weighted = country_spread(df)
rng = np.random.default_rng(SEED); boot = []
groups = {c: g.index.values for c, g in df.groupby("country")}
for _ in range(1000):                                   # resample firms within each country
    idx = np.concatenate([rng.choice(ix, len(ix)) for ix in groups.values()])
    e, w_ = country_spread(df.loc[idx]); boot.append((e["Purchased"] / e["On-site"], w_["Purchased"] / w_["On-site"]))
boot = np.array(boot)
r2_spread = pd.DataFrame({"Purchased": [equal["Purchased"], weighted["Purchased"]], "On-site": [equal["On-site"], weighted["On-site"]]},
                         index=["Every country counted equally", "Countries counted by number of SMEs"])
r2_spread["Purchased / On-site"] = r2_spread["Purchased"] / r2_spread["On-site"]
r2_spread["95% interval, low"] = np.percentile(boot, 2.5, axis=0); r2_spread["95% interval, high"] = np.percentile(boot, 97.5, axis=0)
big5 = list(country_weight.head(5).index)
w = df["weight_country"]
shares_big5 = (100 * w.groupby([df["country"], df["pathway"]]).sum().unstack().pipe(lambda t: t.div(t.sum(axis=1), axis=0))).loc[big5, ["Purchased", "On-site"]].round(1)
print("Pathway shares in the five largest countries (weighted %):"); display(shares_big5)
r2_spread.round(3)

**How to read it.** A ratio above 1 in the last columns means purchasing varies more between countries than on-site generation does. Compare the two rows: if the ratio drops towards 1 in the second row, the statement "country matters more for purchasing" holds when countries are treated as equal units but not when firms are weighted to the EU population.

### Step 3. Refit the models with the weights

The full-predictor models are refitted on the same 20 splits under each weighting. Each firm's survey weight is combined with the class weight so that the three pathways still carry equal total weight. The theme importances are then averaged over test firms using the same survey weight.

In [ ]:
t0 = time.time()
auc_rows = []
for scheme in WEIGHTINGS:
    for i, (tr, te) in enumerate(SPLITS):
        if (scheme, i) not in runs:
            res = run_split(tr, te, "B All", scheme); del res["fitted"]; runs[(scheme, i)] = res
        for model, p in runs[(scheme, i)]["proba"].items():
            auc_rows.append({"Models fitted with": scheme, "Model": model, "split": i,
                             **{f"Test AUC, {ev.lower()}": auc(y.loc[te], p, None if WEIGHTINGS[ev] is None else survey_w(te, ev)) for ev in WEIGHTINGS}})
r2_auc = pd.DataFrame(auc_rows).drop(columns="split").groupby(["Model", "Models fitted with"], sort=False).mean().round(3)
print(f"R2 step 3 took {(time.time() - t0) / 60:.1f} minutes")
print("Average over 20 splits. Columns: how the test firms are weighted when scoring."); r2_auc

In [ ]:
KEY = ["Country", "Investment level", "Support used", "Firm age", "Firm size", "Climate strategy", "Barriers met",
       "Other green actions", "Support wanted", "Green products, jobs, materials", "Turnover and growth", "Sector and offer", "Hiring difficulty"]
def ratio_table(model):
    out = pd.DataFrame(index=KEY)
    for scheme in WEIGHTINGS:
        st = stability(scheme, model).loc[KEY]
        out[(scheme, "ratio")] = st["Ratio median"].round(2); out[(scheme, "splits > 1")] = st["Splits with ratio > 1"]
    out.columns = pd.MultiIndex.from_tuples(out.columns); return out
r2_ratio_xgb, r2_ratio_logit = ratio_table("XGBoost"), ratio_table("Logistic regression")
r2_importance = pd.concat({f"{scheme} | {model}": stability(scheme, model).loc[KEY, ["Purchased", "On-site"]].round(3)
                           for scheme in WEIGHTINGS for model in ["XGBoost", "Logistic regression"]}, axis=1)
print("On-site / purchased importance ratio: median over 20 splits, and in how many of the 20 splits it is above 1")
print("\nXGBoost"); display(r2_ratio_xgb); print("Logistic regression"); r2_ratio_logit

**How to read it.** A theme whose ratio is on the same side of 1 in all 20 splits under all three weightings, in both models, is a robust part of the contrast between the pathways. A theme whose ratio crosses 1 when the weighting changes is not, and the paper must say so.

## R3. Can the model predict firms in a country it has never seen?

In the main analysis, firms from every country are in both the training and the test set, so the model can learn each country's level. That answers the question "how well can we predict another firm in one of these 20 countries?"

A harder question is whether what the model learned transfers to a new country. To test it, each country in turn is held out completely: the model is trained on the other 19 and predicts the held-out one. The country identifier cannot be used here, since the model has never seen the held-out country, so the predictors are the firm characteristics and behaviours only.

For comparison, the same predictors are scored with ordinary 5-fold cross-validation, where countries are mixed.

In [ ]:
t0 = time.time()
NOCOUNTRY = FIRM + BEHAVIOUR
def out_of_fold(folds, make_model, use_sw):
    proba = np.zeros((len(df), 3))
    for tr, te in folds:
        p = make_pre(NOCOUNTRY); Xa = p.fit_transform(df.iloc[tr][NOCOUNTRY]); Xb = p.transform(df.iloc[te][NOCOUNTRY])
        m = make_model()
        m.fit(Xa, y.iloc[tr], sample_weight=compute_sample_weight("balanced", y.iloc[tr])) if use_sw else m.fit(Xa, y.iloc[tr])
        proba[te] = m.predict_proba(Xb)
    return proba
by_country = [(np.where(df["country"] != c)[0], np.where(df["country"] == c)[0]) for c in sorted(df["country"].unique())]
mixed = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(df, y))
p_loco_lg, p_mix_lg = out_of_fold(by_country, lambda: new_logit("B All"), False), out_of_fold(mixed, lambda: new_logit("B All"), False)
p_loco_xg, p_mix_xg = out_of_fold(by_country, new_xgb, True), out_of_fold(mixed, new_xgb, True)
r3 = pd.DataFrame([
    {"Model": "Logistic regression", "Countries mixed (5-fold)": auc(y, p_mix_lg), "Country held out": auc(y, p_loco_lg)},
    {"Model": "XGBoost", "Countries mixed (5-fold)": auc(y, p_mix_xg), "Country held out": auc(y, p_loco_xg)}]).round(3)
r3["Loss when the country is unseen"] = (r3["Countries mixed (5-fold)"] - r3["Country held out"]).round(3)
# AUC inside each held-out country: how well firms are ranked within a country the model never saw
per_country = pd.DataFrame({m: {c: roc_auc_score(y[df["country"] == c], p[(df["country"] == c).values], multi_class="ovr")
                                for c in sorted(df["country"].unique())} for m, p in [("Logistic regression", p_loco_lg), ("XGBoost", p_loco_xg)]}).round(6)
print(f"R3 took {(time.time() - t0) / 60:.1f} minutes")
for m in per_country:
    print("Within-country AUC for held-out countries, %s: median %.3f, lowest %.3f (%s), highest %.3f (%s)" % (
        m, per_country[m].median(), per_country[m].min(), per_country[m].idxmin(), per_country[m].max(), per_country[m].idxmax()))
r3

## R4. The contrast using structural predictors only

Investment, support and the other behavioural predictors are measured at the same time as the outcome and may be part of the same decision. Country, sector, size, age and turnover cannot be a result of the energy decision. If the contrast between the two pathways is real, it should appear in a model that uses only those.

In [ ]:
res_a = run_split(train_idx, test_idx, "A2 Firm + country", MAIN)
r4 = pd.DataFrame({"Purchased": res_a["themes"]["XGBoost"]["Purchased"], "On-site": res_a["themes"]["XGBoost"]["On-site"]})
r4["Ratio, XGBoost"] = r4["On-site"] / r4["Purchased"]
r4["Ratio, logistic"] = res_a["themes"]["Logistic regression"]["On-site"] / res_a["themes"]["Logistic regression"]["Purchased"]
r4 = r4.sort_values("Purchased", ascending=False).round(4)
print("Test AUC, structural predictors only: XGBoost %.3f, logistic regression %.3f" % (
    auc(y.loc[test_idx], res_a["proba"]["XGBoost"]), auc(y.loc[test_idx], res_a["proba"]["Logistic regression"])))
r4

## R5. Does the way SHAP values are added into themes matter?

In the main analysis a theme's importance is the average size of the **net** contribution of its predictors: the SHAP values are added with their signs first, so two predictors pushing in opposite directions cancel. The alternative is to add the predictors' individual importances, which never cancel. If the two versions rank the themes the same way, the choice does not matter.

In [ ]:
B = SETS["B All"]
main = run_split(train_idx, test_idx, "B All", MAIN)           # the main model of the paper
pre, Xtr, Xte, main_lg, main_xg = main["fitted"]
shap_main = shap_by_predictor(main_xg, pre, Xte, B)
ti_main = main["themes"]["XGBoost"]
alt = pd.DataFrame(0.0, index=THEMES, columns=CLASS_NAMES)
per_pred = np.abs(shap_main).mean(axis=0)                    # pathways x predictors
for j, p in enumerate(B):
    alt.loc[theme_of(p)] += per_pred[:, j]
r5 = pd.DataFrame({"Purchased (net, main)": ti_main["Purchased"], "Purchased (no cancelling)": alt["Purchased"],
                   "On-site (net, main)": ti_main["On-site"], "On-site (no cancelling)": alt["On-site"]})
r5["Ratio (net, main)"] = r5["On-site (net, main)"] / r5["Purchased (net, main)"]
r5["Ratio (no cancelling)"] = r5["On-site (no cancelling)"] / r5["Purchased (no cancelling)"]
r5["Predictors in theme"] = [sum(theme_of(p) == t for p in B) for t in r5.index]
r5 = r5.sort_values("On-site (net, main)", ascending=False).round(3)
print("Rank correlation between the two ways of adding up:  Purchased %.2f   On-site %.2f" % (
    spearmanr(ti_main["Purchased"], alt["Purchased"])[0], spearmanr(ti_main["On-site"], alt["On-site"])[0]))
print("Themes with the ratio on the same side of 1 under both: %d of %d" % (
    int(((r5["Ratio (net, main)"] > 1) == (r5["Ratio (no cancelling)"] > 1)).sum()), len(r5)))
r5

In [ ]:
# The list of which predictor belongs to which theme (goes in the paper's appendix)
theme_map = pd.DataFrame({"Predictor": B, "Theme": [theme_of(p) for p in B]}).groupby("Theme")["Predictor"].agg(
    Predictors="count", Columns=lambda s: ", ".join(s)).loc[r5.index]
theme_map

## R6. Do the class weights matter?

In [ ]:
lg_nw = LogisticRegression(max_iter=3000, C=0.1).fit(Xtr, y.loc[train_idx])
xg_nw = new_xgb().fit(Xtr, y.loc[train_idx])
def recalls(m): return " / ".join(f"{v:.0f}" for v in confusion_matrix(y.loc[test_idx], m.predict_proba(Xte).argmax(axis=1), normalize="true").diagonal() * 100)
r6 = pd.DataFrame([{"Model": name, "Class weights": cw_label, "Test AUC": round(auc(y.loc[test_idx], m.predict_proba(Xte)), 3),
                    "Recall % (none / purchased / on-site)": recalls(m)}
                   for name, cw_label, m in [("Logistic regression", "Balanced (main)", main_lg), ("Logistic regression", "No class weights", lg_nw),
                                             ("XGBoost", "Balanced (main)", main_xg), ("XGBoost", "No class weights", xg_nw)]])
r6

## R7. Firm size and firm age are each entered twice. Does it matter?

Size enters as the survey's size class and as the number of employees; age enters in years and in bands. Here the full model is refitted with only one version of each.

In [ ]:
rows = []
for label, drop in [("Both versions (main)", []), ("Continuous only (no size class, no age band)", ["size", "age_band"]),
                    ("Categories only (no employee count, no age in years)", ["log_employees", "log_firm_age"])]:
    cols = [c for c in B if c not in drop]; p = make_pre(cols); Xa = p.fit_transform(df.loc[train_idx, cols]); Xb = p.transform(df.loc[test_idx, cols])
    xg = new_xgb().fit(Xa, y.loc[train_idx], sample_weight=compute_sample_weight("balanced", y.loc[train_idx]))
    lg = new_logit("B All").fit(Xa, y.loc[train_idx])
    ti = theme_importance(shap_by_predictor(xg, p, Xb, cols), cols)
    rows.append({"Predictors": label, "Test AUC, logistic": auc(y.loc[test_idx], lg.predict_proba(Xb)), "Test AUC, XGBoost": auc(y.loc[test_idx], xg.predict_proba(Xb)),
                 "Firm size ratio (XGBoost)": ti.loc["Firm size", "On-site"] / ti.loc["Firm size", "Purchased"],
                 "Firm age ratio (XGBoost)": ti.loc["Firm age", "On-site"] / ti.loc["Firm age", "Purchased"]})
r7 = pd.DataFrame(rows).round(3); r7

## Save the outputs

In [ ]:
flat = lambda t: t.set_axis([" | ".join(map(str, c)) for c in t.columns], axis=1)
pc = per_country["Logistic regression"]
notes = pd.DataFrame({"Item": ["Firms", "Countries", "Firms per country, fewest", "Firms per country, most", "Five largest countries by population weight",
                               "Their share of the population weight %", "Micro firms' share of the population weight %",
                               "Within-country AUC (held out, logistic): median", "lowest", "lowest country", "highest", "highest country",
                               "Rank correlation, net vs no-cancelling importance: purchased", "same: on-site", "Ratios on the same side of 1 under both"],
                      "Value": [len(df), df["country"].nunique(), int(df["country"].value_counts().min()), int(df["country"].value_counts().max()), ", ".join(country_weight.head(5).index),
                                round(country_weight.head(5).sum(), 1), round(100 * df.loc[df["size"].str.startswith("Micro"), "weight_eu27"].sum() / df["weight_eu27"].sum(), 1),
                                round(pc.median(), 3), round(pc.min(), 3), pc.idxmin(), round(pc.max(), 3), pc.idxmax(),
                                round(spearmanr(ti_main["Purchased"], alt["Purchased"])[0], 2), round(spearmanr(ti_main["On-site"], alt["On-site"])[0], 2),
                                int(((r5["Ratio (net, main)"] > 1) == (r5["Ratio (no cancelling)"] > 1)).sum())]})
with pd.ExcelWriter("Day5_results.xlsx") as xl:
    r1.to_excel(xl, sheet_name="R1 repeated splits")
    splits.round(6).to_excel(xl, sheet_name="R1 every split", index=False)
    r1_shap.to_excel(xl, sheet_name="R1 stability XGBoost")
    r1_shap_logit.to_excel(xl, sheet_name="R1 stability logistic")
    r2_weights.to_excel(xl, sheet_name="R2 weights", index=False)
    r2_spread.round(6).to_excel(xl, sheet_name="R2 country spread")
    shares_big5.to_excel(xl, sheet_name="R2 five largest countries")
    r2_auc.reset_index().to_excel(xl, sheet_name="R2 AUC by weighting", index=False)
    flat(r2_ratio_xgb).to_excel(xl, sheet_name="R2 ratios XGBoost")
    flat(r2_ratio_logit).to_excel(xl, sheet_name="R2 ratios logistic")
    flat(r2_importance).to_excel(xl, sheet_name="R2 importances")
    r3.to_excel(xl, sheet_name="R3 unseen country", index=False)
    per_country.to_excel(xl, sheet_name="R3 per country")
    r4.to_excel(xl, sheet_name="R4 structural only")
    r5.to_excel(xl, sheet_name="R5 theme aggregation")
    theme_map.to_excel(xl, sheet_name="R5 theme mapping")
    r6.to_excel(xl, sheet_name="R6 class weights", index=False)
    r7.to_excel(xl, sheet_name="R7 size and age", index=False)
    notes.to_excel(xl, sheet_name="Notes", index=False)
print("Saved Day5_results.xlsx")
try:
    from google.colab import files; files.download("Day5_results.xlsx")
except ImportError:
    pass